In [1]:
# Imports, project paths and the global seed. Prints library versions so the
# notebook's own output records that the pinned stack was used.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr, wasserstein_distance

SEED = 42
rng = np.random.default_rng(SEED)  # all resampling draws from this one generator

# Notebook sits in notebooks/, so the repo root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "outputs" / "tables"
SHAP_DIR = ROOT / "outputs" / "shap"
MODELS = ROOT / "outputs" / "models"

print("root:", ROOT)
print("pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)
print("tables dir exists:", TABLES.exists(), "| shap dir exists:", SHAP_DIR.exists())

root: c:\Fintech-Project\graph_AML_pipeline
pandas 2.2.3 | numpy 1.26.4 | scipy 1.13.1
tables dir exists: True | shap dir exists: True


In [2]:
# Load the global SHAP tables and the per-row SHAP matrices from Notebook 08.
# Prints columns and shapes so the rest of the notebook can reference real names.
m1_global = pd.read_csv(TABLES / "08_m1_global_shap.csv")
m2_global = pd.read_csv(TABLES / "08_m2_global_shap.csv")
shared = pd.read_csv(TABLES / "08_h4_shared_ranks.csv")

m1_shap = np.load(SHAP_DIR / "08_m1_shap.npy")   # (rows, 10 features)
m2_shap = np.load(SHAP_DIR / "08_m2_shap.npy")   # (rows, 26 features)

with open(SHAP_DIR / "08_shap_meta.json") as f:
    meta = json.load(f)

print("m1_global cols:", list(m1_global.columns))
print("m2_global cols:", list(m2_global.columns))
print("shared  cols:", list(shared.columns))
print("shap arrays:", m1_shap.shape, m2_shap.shape)
print("meta keys:", list(meta.keys()))
m1_global.head(3)

m1_global cols: ['Unnamed: 0', 'mean_abs_shap']
m2_global cols: ['Unnamed: 0', 'rank', 'mean_abs_shap', 'family', 'share_%']
shared  cols: ['feature', 'm1_rank', 'm2_rank', 'm1_mean_abs', 'm2_mean_abs']
shap arrays: (10000, 10) (10000, 26)
meta keys: ['m1_base', 'm2_base', 'n_sample', 'n_background', 'm1_trees', 'm2_trees', 'seed', 'population', 'perturbation']


,Unnamed: 0,mean_abs_shap
0,Payment Format,1.245876
1,From Bank,1.003990
2,Amount Paid,0.514667


In [3]:
# Attach feature names to the SHAP arrays from the training feature lists, then check
# that recomputing mean-|SHAP| reproduces Notebook 08's stored tables. Gate: diffs ~0.
with open(MODELS / "06_m1_features.json") as f:
    m1_feats = json.load(f)
with open(MODELS / "07_m2_features.json") as f:
    m2_feats = json.load(f)

print("feature counts:", len(m1_feats), len(m2_feats))
assert len(m1_feats) == m1_shap.shape[1] and len(m2_feats) == m2_shap.shape[1]

# Recompute the global means straight from the per-row matrices.
m1_recomp = pd.Series(np.abs(m1_shap).mean(axis=0), index=m1_feats)
m2_recomp = pd.Series(np.abs(m2_shap).mean(axis=0), index=m2_feats)

# Compare against the stored tables (feature name sits in the unnamed index column).
m1_stored = m1_global.set_index("Unnamed: 0")["mean_abs_shap"]
m2_stored = m2_global.set_index("Unnamed: 0")["mean_abs_shap"]

print("M1 max abs diff:", np.abs(m1_recomp - m1_stored.reindex(m1_recomp.index)).max())
print("M2 max abs diff:", np.abs(m2_recomp - m2_stored.reindex(m2_recomp.index)).max())

SHARED = shared["feature"].tolist()   # the 10 features H4(a) is computed on
print("shared features:", len(SHARED))
print("point estimate rho:", spearmanr(m1_recomp[SHARED], m2_recomp[SHARED]).statistic)

feature counts: 10 26
M1 max abs diff: 2.220446049250313e-16
M2 max abs diff: 8.326672684688674e-17
shared features: 10
point estimate rho: 0.7818181818181817


In [4]:
# Paired row bootstrap for the H4(a) statistic: 2,000 resamples of the 10,000 SHAP rows,
# recomputing mean-|SHAP| and Spearman rho on the 10 shared features each time.
B = 2000
n = m1_shap.shape[0]
shared_idx_m1 = [m1_feats.index(f) for f in SHARED]
shared_idx_m2 = [m2_feats.index(f) for f in SHARED]

m1_abs = np.abs(m1_shap[:, shared_idx_m1])   # pre-slice once, then only resample rows
m2_abs = np.abs(m2_shap[:, shared_idx_m2])

rhos = np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)              # same rows for both models = paired
    rhos[b] = spearmanr(m1_abs[idx].mean(axis=0), m2_abs[idx].mean(axis=0)).statistic

lo, hi = np.percentile(rhos, [2.5, 97.5])
print(f"rho point estimate : {spearmanr(m1_abs.mean(0), m2_abs.mean(0)).statistic:.4f}")
print(f"bootstrap 95% CI   : [{lo:.4f}, {hi:.4f}]   (B={B}, seed={SEED})")
print(f"P(rho <= 0.70)     : {(rhos <= 0.70).mean():.4f}")
print(f"CI excludes 0.70   : {lo > 0.70}")

rho point estimate : 0.7818
bootstrap 95% CI   : [0.7818, 0.7818]   (B=2000, seed=42)
P(rho <= 0.70)     : 0.0000
CI excludes 0.70   : True


In [5]:
# The CI came back zero-width. Check whether rho genuinely never moves, or the
# resampling isn't varying. Also confirm the draws themselves differ.
print("unique rho values :", np.unique(rhos).size)
print("min / max rho     :", rhos.min(), rhos.max())
print("std               :", rhos.std())

# Sanity: two fresh draws from the same generator must not be identical.
check = np.random.default_rng(SEED)
a, b = check.integers(0, n, n), check.integers(0, n, n)
print("draws differ      :", not np.array_equal(a, b), "| overlap:", (a == b).mean())

# How much do the underlying means actually wobble? (rho only sees their ordering.)
one = rng.integers(0, n, n)
print("\nmean-|SHAP| on one resample vs full sample, M1:")
print(pd.DataFrame({"full": m1_abs.mean(0), "resample": m1_abs[one].mean(0)}, index=SHARED))

unique rho values : 1
min / max rho     : 0.7818181818181817 0.7818181818181817
std               : 2.220446049250313e-16
draws differ      : True | overlap: 0.0001

mean-|SHAP| on one resample vs full sample, M1:
                        full  resample
Amount Received     0.375067  0.375084
Amount Paid         0.514667  0.514797
From Bank           1.003990  1.005298
To Bank             0.225370  0.226068
Receiving Currency  0.175651  0.176625
Payment Currency    0.158629  0.159299
Payment Format      1.245876  1.257983
hour                0.202541  0.205126
day_of_week         0.167450  0.169093
is_weekend          0.039025  0.039275


In [6]:
# Registered reported measures alongside H4. Unlike rho (10 shared features only),
# these compare the FULL ranked lists, so displacement out of the ranking is visible.
def rbo(l1, l2, p=0.9):
    """Rank-biased overlap: top-weighted list similarity, tolerates unequal lengths."""
    s1, s2, agree, total = set(), set(), 0.0, 0.0
    for d in range(1, max(len(l1), len(l2)) + 1):
        if d <= len(l1): s1.add(l1[d - 1])
        if d <= len(l2): s2.add(l2[d - 1])
        w = p ** (d - 1)
        agree += w * (len(s1 & s2) / d)   # proportional overlap at depth d
        total += w
    return agree / total

def jaccard_at_k(l1, l2, k):
    a, b = set(l1[:k]), set(l2[:k])
    return len(a & b) / len(a | b)

m1_rank = m1_recomp.sort_values(ascending=False)
m2_rank = m2_recomp.sort_values(ascending=False)
L1, L2 = m1_rank.index.tolist(), m2_rank.index.tolist()

print(f"RBO (p=0.90) full lists : {rbo(L1, L2, 0.9):.4f}")
print(f"RBO (p=0.90) shared only: {rbo([f for f in L1 if f in SHARED], [f for f in L2 if f in SHARED], 0.9):.4f}")
for k in (5, 10, 20):
    print(f"Jaccard@{k:<2}             : {jaccard_at_k(L1, L2, k):.4f}")

# Wasserstein-1 between the normalised importance distributions on shared features.
p1 = m1_recomp[SHARED] / m1_recomp[SHARED].sum()
p2 = m2_recomp[SHARED] / m2_recomp[SHARED].sum()
w1 = wasserstein_distance(np.arange(len(SHARED)), np.arange(len(SHARED)), p1.values, p2.values)
print(f"\nWasserstein-1 (shared, normalised): {w1:.4f}")
print("\nFrom Bank — M1 rank", L1.index("From Bank") + 1, "| M2 rank", L2.index("From Bank") + 1,
      f"| mass {m1_recomp['From Bank']:.4f} -> {m2_recomp['From Bank']:.4f}")

RBO (p=0.90) full lists : 0.5245
RBO (p=0.90) shared only: 0.8054
Jaccard@5              : 0.4286
Jaccard@10             : 0.1765
Jaccard@20             : 0.3043

Wasserstein-1 (shared, normalised): 0.5568

From Bank — M1 rank 2 | M2 rank 14 | mass 1.0040 -> 0.1578


In [7]:
# Persist the rank-agreement measures for the report and the appendix.
pd.DataFrame([
    {"measure": "spearman_rho_shared", "value": 0.7818181818181817, "ci_lo": lo, "ci_hi": hi},
    {"measure": "rbo_p0.9_full", "value": rbo(L1, L2, 0.9), "ci_lo": None, "ci_hi": None},
    {"measure": "rbo_p0.9_shared", "value": rbo([f for f in L1 if f in SHARED], [f for f in L2 if f in SHARED], 0.9), "ci_lo": None, "ci_hi": None},
    {"measure": "jaccard_at_5", "value": jaccard_at_k(L1, L2, 5), "ci_lo": None, "ci_hi": None},
    {"measure": "jaccard_at_10", "value": jaccard_at_k(L1, L2, 10), "ci_lo": None, "ci_hi": None},
    {"measure": "wasserstein1_shared", "value": w1, "ci_lo": None, "ci_hi": None},
]).to_csv(TABLES / "10_rank_agreement.csv", index=False)
print("written:", TABLES / "10_rank_agreement.csv")

written: c:\Fintech-Project\graph_AML_pipeline\outputs\tables\10_rank_agreement.csv


In [8]:
# Load the frozen test probabilities and thresholds, restrict to test-main, and
# rebuild the flag/TP counts. Gate: must reproduce M1 291 TP and M2 216 TP.
m1_probs = pd.read_parquet(MODELS / "m1_test_probs.parquet")
m2_probs = pd.read_parquet(MODELS / "m2_test_probs.parquet")

with open(MODELS / "06_m1_threshold.json") as f:
    thr1 = json.load(f)
with open(MODELS / "07_m2_threshold.json") as f:
    thr2 = json.load(f)

print("m1_probs cols:", list(m1_probs.columns), m1_probs.shape)
print("m2_probs cols:", list(m2_probs.columns), m2_probs.shape)
print("thr1 keys:", list(thr1.keys()))
print("thr2 keys:", list(thr2.keys()))
m1_probs.head(3)

m1_probs cols: ['m1_prob', 'is_main'] (761639, 2)
m2_probs cols: ['m2_prob', 'is_main'] (761639, 2)
thr1 keys: ['threshold', 'val_f1', 'chosen_on', 'n_trees_used', 'num_boost_round', 'early_stopping_rounds']
thr2 keys: ['threshold', 'val_f1', 'chosen_on', 'n_trees_used', 'num_boost_round', 'early_stopping_rounds']


,m1_prob,is_main
812708,0.087262,True
812709,0.864997,True
812710,0.911600,True


In [9]:
# Pull just the label column from the feature store and join it onto the probabilities
# by index. Gate: the rebuilt counts must reproduce M1 291 TP and M2 216 TP.
import pyarrow.parquet as pq

FEATS = ROOT / "data" / "processed" / "05_features_full.parquet"
schema_names = pq.ParquetFile(FEATS).schema.names
label_col = [c for c in schema_names if "launder" in c.lower()][0]
print("label column:", label_col)

labels = pd.read_parquet(FEATS, columns=[label_col])

# Join on index, then restrict to test-main via the stored flag.
df = m1_probs.join(m2_probs["m2_prob"]).join(labels, how="left")
main = df[df["is_main"]].copy()
main["y"] = main[label_col].astype(int)

t1, t2 = thr1["threshold"], thr2["threshold"]
main["f1"] = main["m1_prob"] >= t1          # M1 flagged
main["f2"] = main["m2_prob"] >= t2          # M2 flagged

print(f"\nthresholds: M1 {t1:.6f} | M2 {t2:.6f}")
print("test-main rows:", len(main), "| illicit:", int(main['y'].sum()))
print("M1 flagged:", int(main['f1'].sum()), "| TP:", int((main['f1'] & (main['y'] == 1)).sum()))
print("M2 flagged:", int(main['f2'].sum()), "| TP:", int((main['f2'] & (main['y'] == 1)).sum()))

label column: Is Laundering

thresholds: M1 0.957998 | M2 0.953385
test-main rows: 760531 | illicit: 906
M1 flagged: 3646 | TP: 291
M2 flagged: 2720 | TP: 216


In [10]:
# McNemar on paired correctness over test-main, plus the registered error-overlap 2x2.
# Exact binomial test (not chi-square) since we care about the discordant pairs directly.
from scipy.stats import binomtest

main["c1"] = main["f1"] == (main["y"] == 1)   # M1 correct on this row
main["c2"] = main["f2"] == (main["y"] == 1)   # M2 correct on this row

n11 = int((main["c1"] & main["c2"]).sum())     # both correct
n10 = int((main["c1"] & ~main["c2"]).sum())    # M1 only  -> M2 lost it
n01 = int((~main["c1"] & main["c2"]).sum())    # M2 only  -> M2 gained it
n00 = int((~main["c1"] & ~main["c2"]).sum())   # both wrong

print("Error-overlap 2x2 (all test-main):")
print(pd.DataFrame([[n11, n10], [n01, n00]],
                   index=["M2 correct", "M2 wrong"], columns=["M1 correct", "M1 wrong"]))

mc = binomtest(min(n10, n01), n10 + n01, 0.5)
print(f"\nMcNemar exact: n10={n10}, n01={n01}, discordant={n10+n01}, p={mc.pvalue:.3e}")

# The same test on illicit rows only -- this is the detection question.
ill = main[main["y"] == 1]
d10 = int((ill["c1"] & ~ill["c2"]).sum())      # M1 caught, M2 missed
d01 = int((~ill["c1"] & ill["c2"]).sum())      # M2 caught, M1 missed
mc_ill = binomtest(min(d10, d01), d10 + d01, 0.5)
print(f"\nIllicit rows only: M1-only={d10}, M2-only={d01}, p={mc_ill.pvalue:.3e}")

Error-overlap 2x2 (all test-main):
            M1 correct  M1 wrong
M2 correct      754383      2178
M2 wrong          2954      1016

McNemar exact: n10=2178, n01=2954, discordant=5132, p=2.253e-27

Illicit rows only: M1-only=170, M2-only=95, p=4.769e-06


In [11]:
# Persist the paired-comparison results.
pd.DataFrame([
    {"test": "mcnemar_all_rows", "m1_only": n10, "m2_only": n01, "p": mc.pvalue},
    {"test": "mcnemar_illicit_only", "m1_only": d10, "m2_only": d01, "p": mc_ill.pvalue},
    {"test": "overlap_both_correct", "m1_only": None, "m2_only": None, "p": None, "n": n11},
    {"test": "overlap_both_wrong", "m1_only": None, "m2_only": None, "p": None, "n": n00},
]).to_csv(TABLES / "10_paired_tests.csv", index=False)
print("written:", TABLES / "10_paired_tests.csv")

written: c:\Fintech-Project\graph_AML_pipeline\outputs\tables\10_paired_tests.csv


In [13]:
# Item 1: exact confounded-feature share of M2's graph SHAP mass.
# Four of eight vertex features are currency-confounded; they appear as from_/to_ pairs.
CONFOUNDED_STEMS = ["in_weighted", "out_weighted", "net_flow", "pagerank"]

graph_feats = [f for f in m2_feats if f.startswith(("from_g_", "to_g_"))]
conf_feats = [f for f in graph_feats if any(s in f for s in CONFOUNDED_STEMS)]

graph_mass = m2_recomp[graph_feats].sum()
conf_mass = m2_recomp[conf_feats].sum()
total_mass = m2_recomp.sum()

print(f"graph features: {len(graph_feats)} | confounded: {len(conf_feats)}")
print(f"confounded share OF GRAPH mass : {100*conf_mass/graph_mass:.2f}%")
print(f"confounded share OF TOTAL mass : {100*conf_mass/total_mass:.2f}%")
print(f"graph share of total (H2 check): {100*graph_mass/total_mass:.2f}% ")

graph features: 16 | confounded: 8
confounded share OF GRAPH mass : 60.92%
confounded share OF TOTAL mass : 32.17%
graph share of total (H2 check): 52.81% 


In [14]:
# Persist the post-hoc confounded-mass figures alongside the H2 check.
pd.DataFrame([
    {"measure": "graph_share_of_total_pct", "value": 100*graph_mass/total_mass, "note": "registered H2"},
    {"measure": "confounded_share_of_graph_pct", "value": 100*conf_mass/graph_mass, "note": "post-hoc"},
    {"measure": "confounded_share_of_total_pct", "value": 100*conf_mass/total_mass, "note": "post-hoc"},
    {"measure": "n_graph_features", "value": len(graph_feats), "note": ""},
    {"measure": "n_confounded_features", "value": len(conf_feats), "note": "4 stems x 2 endpoints"},
]).to_csv(TABLES / "10_confounded_mass.csv", index=False)
print("written:", TABLES / "10_confounded_mass.csv")

written: c:\Fintech-Project\graph_AML_pipeline\outputs\tables\10_confounded_mass.csv
